<a href="https://colab.research.google.com/github/khalidkhankakar/Hands-on-Machine-Learning/blob/master/chapter_10_ANN/neural_network_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Linear Reqression with Pytorch

In [4]:
import torch
from sklearn.datasets import fetch_california_housing
X, y = fetch_california_housing(return_X_y=True)
X.shape, y.shape


((20640, 8), (20640,))

In [5]:
from sklearn.model_selection import train_test_split
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, random_state=42)
X_train, X_dev, y_train, y_dev = train_test_split(X_train_full, y_train_full, random_state=42)
X_train.shape, y_train.shape, X_test.shape, y_test.shape, X_dev.shape, y_dev.shape

((11610, 8), (11610,), (5160, 8), (5160,), (3870, 8), (3870,))

In [6]:
X_train = torch.FloatTensor(X_train)
X_dev = torch.FloatTensor(X_dev)
X_test = torch.FloatTensor(X_test)
means = X_train.mean(dim=0, keepdim=True)
stds = X_train.std(dim=0, keepdim=True)
X_train = (X_train - means) / stds
X_test = (X_test- means) / stds
X_dev = (X_dev - means) / stds

In [7]:
y_train= torch.FloatTensor(y_train.reshape(-1, 1))
y_dev= torch.FloatTensor(y_dev.reshape(-1, 1))
y_test= torch.FloatTensor(y_test.reshape(-1, 1))

In [8]:
# Our parameter for network
n_features = X_train.shape[1]
weights = torch.randn((n_features, 1), requires_grad=True)
bias = torch.tensor(0., requires_grad=True)

In [9]:
# let's train the model
learning_rate = 0.01
n_epochs = 20

for epoch in range(n_epochs):
  y_pred = X_train @ weights + bias
  loss = ((y_pred - y_train)**2).mean()
  loss.backward()
  with torch.no_grad():
    bias -= learning_rate * bias.grad
    weights -= learning_rate * weights.grad
    bias.grad.zero_()
    weights.grad.zero_()
  print(f"Epoch {epoch + 1}/{n_epochs}, Loss {loss.item()}")

Epoch 1/20, Loss 13.059897422790527
Epoch 2/20, Loss 12.485189437866211
Epoch 3/20, Loss 11.94035816192627
Epoch 4/20, Loss 11.423662185668945
Epoch 5/20, Loss 10.933476448059082
Epoch 6/20, Loss 10.468273162841797
Epoch 7/20, Loss 10.026622772216797
Epoch 8/20, Loss 9.607195854187012
Epoch 9/20, Loss 9.208740234375
Epoch 10/20, Loss 8.830076217651367
Epoch 11/20, Loss 8.470105171203613
Epoch 12/20, Loss 8.127799034118652
Epoch 13/20, Loss 7.802184104919434
Epoch 14/20, Loss 7.492354869842529
Epoch 15/20, Loss 7.197454929351807
Epoch 16/20, Loss 6.916682243347168
Epoch 17/20, Loss 6.6492815017700195
Epoch 18/20, Loss 6.394545555114746
Epoch 19/20, Loss 6.151805877685547
Epoch 20/20, Loss 5.920434474945068


In [10]:
X_new = X_test[:3]
with torch.no_grad():
  y_pred = X_new @ weights + bias

In [11]:
y_pred

tensor([[-1.1286],
        [ 0.2140],
        [ 0.4381]])

Linear Regression with torch high level API

In [12]:
import torch.nn as nn
model = nn.Linear(in_features= n_features, out_features = 1)

In [13]:
# parameters
model.bias, model.weight

(Parameter containing:
 tensor([-0.2380], requires_grad=True),
 Parameter containing:
 tensor([[-0.2529,  0.2366,  0.2604,  0.2529, -0.1054, -0.0820, -0.2909,  0.2409]],
        requires_grad=True))

In [14]:
for param in model.parameters():
  print(param)
for named_param in model.named_parameters():
  print(named_param[0], named_param[1])

Parameter containing:
tensor([[-0.2529,  0.2366,  0.2604,  0.2529, -0.1054, -0.0820, -0.2909,  0.2409]],
       requires_grad=True)
Parameter containing:
tensor([-0.2380], requires_grad=True)
weight Parameter containing:
tensor([[-0.2529,  0.2366,  0.2604,  0.2529, -0.1054, -0.0820, -0.2909,  0.2409]],
       requires_grad=True)
bias Parameter containing:
tensor([-0.2380], requires_grad=True)


In [15]:
# here model is just normal.
# so model is not train yet. it will make terriable predications
# Note: when call the this torch will internall call forward() method Which is just as see in previous example. X @ self.weight + self.bias
model(X_train[:3])

tensor([[-1.2539],
        [-0.5059],
        [-0.4015]], grad_fn=<AddmmBackward0>)

In [16]:
# optimizer: In short it updates the learnable parameters values like weights and bias. You can choose different kinds algorithm to perform backpropagation
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()

In [17]:
def train_bgd(model, optimizer, criterion, X_train, y_train, n_epochs):
  for epoch in range(n_epochs):
    y_pred = model(X_train)
    loss = criterion(y_pred, y_train)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    print(f'{epoch}/{n_epochs}, loss: {loss.item()}')

In [18]:
train_bgd(model, optimizer, mse, X_train, y_train, 20)

0/20, loss: 7.481278896331787
1/20, loss: 7.202644348144531
2/20, loss: 6.935547828674316
3/20, loss: 6.679497718811035
4/20, loss: 6.434024333953857
5/20, loss: 6.1986823081970215
6/20, loss: 5.9730424880981445
7/20, loss: 5.75669527053833
8/20, loss: 5.549248218536377
9/20, loss: 5.3503289222717285
10/20, loss: 5.159575939178467
11/20, loss: 4.976648807525635
12/20, loss: 4.801219940185547
13/20, loss: 4.632974147796631
14/20, loss: 4.471612453460693
15/20, loss: 4.3168463706970215
16/20, loss: 4.168402194976807
17/20, loss: 4.026015758514404
18/20, loss: 3.8894360065460205
19/20, loss: 3.758420705795288


Implement MLP with low level API

In [19]:
# input layer weights and bias
w1 = torch.randn((n_features, 64), requires_grad=True)
b1 = torch.zeros(64, requires_grad=True)

# second layer weights and bias
w2 = torch.randn((64, 32), requires_grad=True)
b2 = torch.zeros(32, requires_grad=True)

# Output layer weights and bias
w3 = torch.randn((32, 1), requires_grad=True)
b3 = torch.zeros(1, requires_grad=True)

parameters = [w1, b1, w2, b2, w3, b3]
print(sum(p.nelement() for p in parameters))

2689


In [20]:
n_epochs = 50
for epoch in range(n_epochs):
  # forward pass
  x = X_train @ w1 + b1
  x = torch.relu(x)
  x = x @ w2 + b2
  x = torch.relu(x)
  y_pred = x @ w3 + b3
  loss = ((y_pred - y_train)**2).mean()


  # backward pass
  for p in parameters:
    p.grad= None
  loss.backward()

  # update
  lr = 0.001
  with torch.no_grad():
    for p in parameters:
      p.data -= lr * p.grad

  print(f'{epoch}/{n_epochs}, loss: {loss.item():.4f}')


0/50, loss: 6099.2095
1/50, loss: 66439.8047
2/50, loss: 5688.7690
3/50, loss: 75.8484
4/50, loss: 61.9109
5/50, loss: 52.4199
6/50, loss: 45.7891
7/50, loss: 40.7140
8/50, loss: 36.6560
9/50, loss: 33.3209
10/50, loss: 30.5097
11/50, loss: 28.1255
12/50, loss: 26.0889
13/50, loss: 24.3276
14/50, loss: 22.7879
15/50, loss: 21.4304
16/50, loss: 20.2281
17/50, loss: 19.1576
18/50, loss: 18.2004
19/50, loss: 17.3431
20/50, loss: 16.5729
21/50, loss: 15.8724
22/50, loss: 15.2321
23/50, loss: 14.6471
24/50, loss: 14.1113
25/50, loss: 13.6194
26/50, loss: 13.1670
27/50, loss: 12.7484
28/50, loss: 12.3603
29/50, loss: 11.9998
30/50, loss: 11.6633
31/50, loss: 11.3489
32/50, loss: 11.0570
33/50, loss: 10.7853
34/50, loss: 10.5297
35/50, loss: 10.2893
36/50, loss: 10.0631
37/50, loss: 9.8490
38/50, loss: 9.6464
39/50, loss: 9.4545
40/50, loss: 9.2719
41/50, loss: 9.0982
42/50, loss: 8.9334
43/50, loss: 8.7764
44/50, loss: 8.6270
45/50, loss: 8.4847
46/50, loss: 8.3489
47/50, loss: 8.2193
48/50,

In [21]:
# predications
with torch.no_grad():
  x = X_dev[:10] @ w1 + b1
  x = torch.relu(x)
  x = x @ w2 + b2
  x = torch.relu(x)
  y_pred = x @ w3 + b3

print(y_pred)


tensor([[ 0.0691],
        [ 0.2323],
        [-2.0655],
        [ 3.4636],
        [-0.7910],
        [ 0.1337],
        [-1.3140],
        [ 0.0691],
        [-0.1450],
        [-0.5356]])


MLP with high level API

In [22]:
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(n_features, 50),
    nn.ReLU(),
    nn.Linear(50, 40),
    nn.ReLU(),
    nn.Linear(40, 1)
)

learning_rate = 0.1
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
mse = nn.MSELoss()
train_bgd(model, optimizer, mse, X_train, y_train, 50)

0/50, loss: 5.045480251312256
1/50, loss: 2.0523123741149902
2/50, loss: 1.0039883852005005
3/50, loss: 0.8570139408111572
4/50, loss: 0.7740675210952759
5/50, loss: 0.7225847244262695
6/50, loss: 0.6893726587295532
7/50, loss: 0.6669032573699951
8/50, loss: 0.6507738828659058
9/50, loss: 0.6383934020996094
10/50, loss: 0.6281993389129639
11/50, loss: 0.6193399429321289
12/50, loss: 0.6113173365592957
13/50, loss: 0.6038705706596375
14/50, loss: 0.5968307852745056
15/50, loss: 0.5901119112968445
16/50, loss: 0.5836468935012817
17/50, loss: 0.5774063467979431
18/50, loss: 0.5713554620742798
19/50, loss: 0.565444827079773
20/50, loss: 0.5596969127655029
21/50, loss: 0.5540881752967834
22/50, loss: 0.5486184358596802
23/50, loss: 0.5432835817337036
24/50, loss: 0.5380954146385193
25/50, loss: 0.533028244972229
26/50, loss: 0.5280747413635254
27/50, loss: 0.5232481956481934
28/50, loss: 0.51854008436203
29/50, loss: 0.5139478445053101
30/50, loss: 0.5094838738441467
31/50, loss: 0.50514501

In [23]:
# DataLoader: it load the batches of data of the deirsed size and suffle them effectively
from torch.utils.data import DataLoader, TensorDataset
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [24]:
# hardware acceleration
if torch.cuda.is_available():
 device = "cuda"
elif torch.backends.mps.is_available():
 device = "mps"
else:
 device = "cpu"
device

'cuda'

In [25]:
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(n_features, 50),
    nn.ReLU(),
    nn.Linear(50, 40),
    nn.ReLU(),
    nn.Linear(40, 1)
)

model = model.to(device)

In [26]:
def train(model, optimizer, criterion, train_loader, n_epochs):
  model.train()
  for epoch in range(n_epochs):
    total_loss = 0.
    for X_batch, y_batch in train_loader:
      X_batch, y_batch = X_batch.to(device), y_batch.to(device)
      y_pred= model(X_batch)
      loss = criterion(y_pred, y_batch)
      total_loss += loss.item()
      loss.backward()
      optimizer.step()
      optimizer.zero_grad()

    mean_loss = total_loss / len(train_loader)
    print(f'{epoch + 1}/ {n_epochs}, loss: {mean_loss:.4f}')


In [27]:
train(model, optimizer, mse, train_loader, 20)

1/ 50, loss: 5.0460
2/ 50, loss: 5.0443
3/ 50, loss: 5.0456
4/ 50, loss: 5.0457
5/ 50, loss: 5.0450
6/ 50, loss: 5.0452
7/ 50, loss: 5.0449
8/ 50, loss: 5.0449
9/ 50, loss: 5.0459
10/ 50, loss: 5.0460
11/ 50, loss: 5.0456
12/ 50, loss: 5.0449
13/ 50, loss: 5.0461
14/ 50, loss: 5.0452
15/ 50, loss: 5.0456
16/ 50, loss: 5.0454
17/ 50, loss: 5.0455
18/ 50, loss: 5.0460
19/ 50, loss: 5.0451
20/ 50, loss: 5.0456
21/ 50, loss: 5.0445
22/ 50, loss: 5.0445
23/ 50, loss: 5.0471
24/ 50, loss: 5.0463
25/ 50, loss: 5.0452
26/ 50, loss: 5.0461
27/ 50, loss: 5.0459
28/ 50, loss: 5.0455
29/ 50, loss: 5.0452
30/ 50, loss: 5.0458
31/ 50, loss: 5.0458
32/ 50, loss: 5.0455
33/ 50, loss: 5.0456
34/ 50, loss: 5.0462
35/ 50, loss: 5.0459
36/ 50, loss: 5.0468
37/ 50, loss: 5.0454
38/ 50, loss: 5.0451
39/ 50, loss: 5.0455
40/ 50, loss: 5.0451
41/ 50, loss: 5.0458
42/ 50, loss: 5.0451
43/ 50, loss: 5.0455
44/ 50, loss: 5.0456
45/ 50, loss: 5.0451
46/ 50, loss: 5.0454
47/ 50, loss: 5.0449
48/ 50, loss: 5.0448
4

In [28]:
def evaluate(model, data_loader,metric_fn, aggergate_fn=torch.mean ):
  model.eval()
  metrics = []
  with torch.no_grad():
    for X_batch, y_batch in data_loader:
      X_batch, y_batch= X_batch.to(device), y_batch.to(device)
      y_pred = model(X_batch)
      metric = metric_fn(y_pred, y_batch)
      metrics.append(metric)
  return aggergate_fn(torch.stack(metrics))

In [29]:
dev_dataset = TensorDataset(X_dev, y_dev)
dev_loader = DataLoader(dev_dataset, batch_size=32)
dev_mse = evaluate(model, dev_loader, mse)
dev_mse

tensor(4.8635, device='cuda:0')

In [33]:
def rmse(y_pred, y_true):
  return ((y_pred -y_true).pow(2)).mean().sqrt()
evaluate(model, dev_loader,rmse)

tensor(2.1945, device='cuda:0')

In [34]:
%pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 51.9 MB/s eta 0:00:00


torchmetrics

In [36]:
import torchmetrics
def evaluate(model, data_loader,metric ):
  model.eval()
  metric.reset()
  with torch.no_grad():
    for X_batch, y_batch in data_loader:
      X_batch, y_batch= X_batch.to(device), y_batch.to(device)
      y_pred = model(X_batch)
      metric.update(y_pred, y_batch)

  return metric.compute()

In [37]:
rmse = torchmetrics.MeanSquaredError(squared=False).to(device)
evaluate(model, dev_loader, rmse)

tensor(2.2054, device='cuda:0')

Building Nonsequential Model Using Custom Modules


In [39]:
# let's wide and deep model
class WideAndDeepV1(nn.Module):
  def __init__(self, n_features):
    super().__init__()
    self.deep_stack = nn.Sequential(
        nn.Linear(n_features, 50), nn.ReLU(),
        nn.Linear(50, 40), nn.ReLU()
    )
    self.output_layer = nn.Linear(40 + n_features, 1)

  def forward(self, X):
    deep_output = self.deep_stack(X)
    wide_and_deep = torch.concat([X, deep_output], dim=1)
    return self.output_layer(wide_and_deep)


In [42]:
torch.manual_seed(42)
model = WideAndDeepV1(n_features).to(device)
train(model, optimizer, mse,train_loader,  20)

1/ 20, loss: 5.7924
2/ 20, loss: 5.7924
3/ 20, loss: 5.7916
4/ 20, loss: 5.7921
5/ 20, loss: 5.7915
6/ 20, loss: 5.7914
7/ 20, loss: 5.7926
8/ 20, loss: 5.7928
9/ 20, loss: 5.7924
10/ 20, loss: 5.7915
11/ 20, loss: 5.7931
12/ 20, loss: 5.7920
13/ 20, loss: 5.7924
14/ 20, loss: 5.7921
15/ 20, loss: 5.7923
16/ 20, loss: 5.7929
17/ 20, loss: 5.7917
18/ 20, loss: 5.7923
19/ 20, loss: 5.7910
20/ 20, loss: 5.7910


In [43]:
evaluate(model,dev_loader, rmse)

tensor(2.3629, device='cuda:0')

In [47]:
# let's wide and deep model
class WideAndDeepV2(nn.Module):
  def __init__(self, n_features):
    super().__init__()
    self.deep_stack = nn.Sequential(
        nn.Linear(n_features-2, 50), nn.ReLU(),
        nn.Linear(50, 40), nn.ReLU()
    )
    self.output_layer = nn.Linear(40 + 5, 1)

  def forward(self, X):
    X_wide = X[:, :5]
    X_deep = X[:, 2:]
    deep_output = self.deep_stack(X_deep)
    wide_and_deep = torch.concat([X_wide, deep_output], dim=1)
    return self.output_layer(wide_and_deep)


In [48]:
torch.manual_seed(42)
model = WideAndDeepV2(n_features).to(device)
train(model, optimizer, mse,train_loader,  20)

1/ 20, loss: 5.7440
2/ 20, loss: 5.7453
3/ 20, loss: 5.7447
4/ 20, loss: 5.7441
5/ 20, loss: 5.7443
6/ 20, loss: 5.7449
7/ 20, loss: 5.7450
8/ 20, loss: 5.7445
9/ 20, loss: 5.7451
10/ 20, loss: 5.7449
11/ 20, loss: 5.7444
12/ 20, loss: 5.7436
13/ 20, loss: 5.7461
14/ 20, loss: 5.7442
15/ 20, loss: 5.7445
16/ 20, loss: 5.7446
17/ 20, loss: 5.7449
18/ 20, loss: 5.7453
19/ 20, loss: 5.7449
20/ 20, loss: 5.7450


In [49]:
evaluate(model,dev_loader, rmse)

tensor(2.3386, device='cuda:0')

In [51]:
class WideAndDeepV3(nn.Module):
  def __init__(self, n_features):
    super().__init__()
    self.deep_stack = nn.Sequential(
        nn.Linear(n_features-2, 50), nn.ReLU(),
        nn.Linear(50, 40), nn.ReLU()
    )
    self.output_layer = nn.Linear(40 + 5, 1)

  def forward(self, X_wide, X_deep):
    deep_output = self.deep_stack(X_deep)
    wide_and_deep = torch.concat([X_wide, deep_output], dim=1)
    return self.output_layer(wide_and_deep)


In [50]:
train_data_wd = TensorDataset(X_train[:, :5], X_train[:, 2:], y_train)
train_loader_wd = DataLoader(train_data_wd, batch_size=32, shuffle=True)


In [ ]:
# for X_batch_wide, X_batch_deep, y_batch in train_loader_wd:
#   X_batch_wide = X_batch_wide.to(device)
#   X_batch_deep = X_batch_deep.to(device)
#   y_batch = y_batch.to(device)

#   y_pred = model(X_batch_wide)
